# Festival Intelligence Terminal - Data Collection

This notebook demonstrates how to collect and process data for the Festival Intelligence Terminal.

## Data Sources
- MusicBrainz: Canonical artist identity
- setlist.fm: Concert history
- Ticketmaster: Future events
- Wikimedia: Pageview attention
- YouTube: Video engagement
- GDELT: News sentiment
- NWS/NOAA: Weather data
- BTS: Air travel data

In [ ]:
import os
import sys
sys.path.append('..')

from pipelines.musicbrainz import MusicBrainzClient, normalize_artist_name
from pipelines.setlistfm import SetlistFMClient, extract_concert_data
from pipelines.ticketmaster import TicketmasterClient, extract_event_data
from pipelines.entity_resolution import EntityResolver, ArtistMapping
from contracts.festivals import INITIAL_FESTIVALS

import polars as pl
from datetime import datetime

## Initialize API Clients

In [ ]:
# Initialize MusicBrainz client
mb_client = MusicBrainzClient(
    user_agent="festival-intelligence/1.0 (your-email@example.com)"
)

# Initialize setlist.fm client (requires API key)
# setlist_client = SetlistFMClient(api_key=os.getenv('SETLISTFM_API_KEY'))

# Initialize Ticketmaster client (requires API key)
# tm_client = TicketmasterClient(api_key=os.getenv('TICKETMASTER_API_KEY'))

# Initialize entity resolver
resolver = EntityResolver()

## Search for Artists

In [ ]:
# Search for an artist
artist_name = "The Weeknd"
results = mb_client.search_artist(artist_name, limit=5)

print(f"Found {len(results)} results for '{artist_name}':")
for result in results:
    print(f"  - {result.get('name')} (ID: {result.get('id')})")

## Fetch Artist Details

In [ ]:
# Get detailed artist data
artist_id = "f7d31c5f-c712-4603-8eb4-3b0b846c4f3c"  # The Weeknd
artist_data = mb_client.get_artist(artist_id)

if artist_data:
    print(f"Name: {artist_data.get('name')}")
    print(f"Country: {artist_data.get('country')}")
    print(f"Type: {artist_data.get('type')}")
    print(f"Aliases: {[a.get('name') for a in artist_data.get('aliases', [])]}")

## Fetch Artist Releases

In [ ]:
# Get recent releases
releases = mb_client.get_artist_releases(artist_id, release_type="album")

print(f"Recent albums ({len(releases)}):")
for release in releases[:5]:
    print(f"  - {release.get('title')} ({release.get('date')})")

## Entity Resolution

In [ ]:
# Add artist mapping to resolver
if artist_data:
    mapping = ArtistMapping(
        musicbrainz_id=artist_data.get('id'),
        normalized_name=normalize_artist_name(artist_data.get('name')),
        name=artist_data.get('name'),
        aliases=[a.get('name') for a in artist_data.get('aliases', [])],
        country=artist_data.get('country'),
        confidence=1.0,
    )
    resolver.add_mapping(mapping)
    print(f"Added mapping for {artist_data.get('name')}")

## Festival Lineup Data

In [ ]:
# Load initial festival data
festivals_df = pl.DataFrame(INITIAL_FESTIVALS)
print(f"Loaded {len(festivals_df)} festivals:")
print(festivals_df.select(['id', 'name', 'city', 'state']))

## Save Data to Warehouse

In [ ]:
# Create warehouse directories
os.makedirs('../warehouse/raw', exist_ok=True)
os.makedirs('../warehouse/normalized', exist_ok=True)

# Save festivals to Parquet
festivals_df.write_parquet('../warehouse/raw/festivals.parquet')
print("Saved festivals to warehouse/raw/festivals.parquet")

## Next Steps

1. Collect historical lineup data for each festival
2. Fetch artist data for all lineup artists
3. Integrate attention metrics (YouTube, Wikimedia, GDELT)
4. Calculate momentum scores and Booking Value Index
5. Run tour prediction models
6. Generate festival comparisons